In [ ]:
import os
import re
import warnings
import random
import json

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import requests

from tqdm import tqdm
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.metrics.pairwise import cosine_similarity

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from imblearn.over_sampling import SMOTE, ADASYN, SVMSMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.ensemble import BalancedRandomForestClassifier

from fairlearn.metrics import MetricFrame, demographic_parity_difference, equalized_odds_difference

from sentence_transformers import SentenceTransformer, util, CrossEncoder

import torch

from utils.feature_engineering import *
from utils.data_cleaning import *

warnings.filterwarnings('ignore')

### Load the data

In [ ]:
pd.set_option('display.max_columns', None)  
df = pd.read_excel('../Dataset_2.0_Akkodis.xlsx')

## Remove rows

### Drop duplicates

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)

### Clean the columns' names

In [ ]:
df = clean_dataframe_columns(df)

### Create new IDs and separate different people with duplicating IDs

In [ ]:
invariant_columns = [
    "ID",
    "Sex",
    "Job Title Hiring",
    "Study Area.1",
    "Assumption Headquarters",
    "Year of insertion",
    "Age Range",
    "Study area",
    "Study Title",
    "Years Experience",
    "Residence",
]
df = split_duplicate_ids_by_invariant_columns(df, invariant_columns)

### Extract `number_of_searches` column

In [ ]:
df['number_of_searches'] = pd.to_numeric(
    df['linked_search__key'].str.split('.', n=1, expand=True)[1], 
    errors='coerce'
)

### Remove irrelevant columns

In [ ]:
df = df.drop(columns=['linked_search__key', 'Year of Recruitment']) 

### Remove candidates in first stages

In [ ]:
df = remove_initial_stage_candidates(df)

### Removal of Candidates with Inconsistent Final Outcomes

In [ ]:
state_order = ['imported', 'first contact', 'in selection', 'qm', 'economic proposal', 'vivier', 'hired']
event_order = ['cv request', 'contact note', 'hr interview', 'bm interview', 'technical interview', 
               'qualification meeting', 'economic proposal', 'candidate notification']

grouped = df.groupby('ID', group_keys=False).apply(sort_group, state_order=state_order, event_order=event_order)

df = grouped.reset_index(drop=True)

In [ ]:
feedbacks_to_remove = [
        'OK (other candidate)', 
        'KO (lost availability)', 
        'OK (hired)', 
        'OK (waiting for departure)', 
        'KO (opportunity closed)', 
        'KO (retired)', 
        'KO (ral)', 
        'KO (proposed renunciation)'
    ]


df = remove_not_hired_valid_candidates(df, state_order=state_order, event_order=event_order, feedbacks_to_remove=feedbacks_to_remove)

In [ ]:
states_to_drop = ['vivier', 'economic proposal']

print("Before filtering:")
for c in list(set(df['Candidate State'])):
    print(f"{c}: {len(df[df['Candidate State']==c])} rows")
total_ids_before = df['ID'].nunique()

df = df[~df['Candidate State'].isin(states_to_drop)]

print("\nAfter filtering:")
for c in list(set(df['Candidate State'])):
    print(f"{c}: {len(df[df['Candidate State']==c])} rows")

total_ids_after = df['ID'].nunique()
print(f"Total IDs before cleaning: {total_ids_before}")
print(f"Total IDs after cleaning: {total_ids_after}")
print(f"Total IDs removed: {total_ids_before - total_ids_after}")

## Preprocess columns

### Ral Mapping

In [ ]:
ral_mapping = {
    '- 20 K': 19000,
    '- 20K': 19000,
    '20-22 K': 21000,
    '20-22K': 21000,
    '22-24 K': 23000,
    '22-24K': 23000,
    '24-26 K': 25000,
    '24-26K': 25000,
    '26-28 K': 27000,
    '26-28K': 27000,
    '28-30 K': 29000,
    '28-30K': 29000,
    '30-32 K': 31000,
    '30-32K': 31000,
    '32-34 K': 33000,
    '32-34K': 33000,
    '34-36 K': 35000,
    '34-36K': 35000,
    '36-38 K': 37000,
    '36-38K': 37000,
    '38-40 K': 39000,
    '38-40K': 39000,
    '40-42 K': 41000,
    '40-42K': 41000,
    '42-44 K': 43000,
    '42-44K': 43000,
    '44-46 K': 45000,
    '44-46K': 45000,
    '46-48 K': 47000,
    '46-48K': 47000,
    '48-50 K': 49000,
    '48-50K': 49000,
    '+ 50 K': 55000,
    '+50K': 55000,
    '20K': 20000,
    'Not available': None,
    'Not Avail.': None,
    np.nan: None
}

ral_columns = ['Expected Ral', 'Minimum Ral', 'Ral Maximum', 'Current Ral']

for col in ral_columns:
    if col in df.columns:
        df[col] = df[col].astype(str).map(ral_mapping)
    else:
        print(f"Warning: Column '{col}' not found in the DataFrame.")

### Overall mapping

In [ ]:
print(f"The unique values of column `Overall` are {set(df['Overall'])}")

In [ ]:
score_mapping = {
    '1 - Low': 1,
    '2 - Medium': 2,
    '3 - High': 3,
    '4 - Top': 4,
    '~ 1 - Low': 1,
    '~ 2 - Medium': 2,
    '~ 3 - High': 3,
    '~ 4 - Top': 4
}
df['Overall'] = df['Overall'].map(score_mapping)

### Make the `Protected category` column boolean

In [ ]:
df['Protected category'] = df['Protected category'].apply(lambda x: True if 'article' in str(x).lower() else False)

### Remove invalid values from `Job Title Hiring`

In [ ]:
df['Job Title Hiring'] = df['Job Title Hiring'].replace('???', None)

### Aggregate Records

In [ ]:
def clean_text(text):
    if not isinstance(text, str) or not text.strip():
        return None
    if text.startswith('o '):
        text = text[1:].strip()
    text = re.sub(r'^[\-\•\*]+\s*', '', text.strip())

    text = re.sub(r'\s+', ' ', text)

    text = text.lower().strip()
    return text
    

for col in ['Candidate Profile','Last Role','Job Description','Candidate Profile']:
    df[col] = df[col].apply(clean_text)

In [ ]:
def find_differences_by_id(df):
    ignore_columns = {'Job Description','event_feedback', 'event_type__val', "Overall", "Minimum Ral",'Ral Maximum', "Technical Skills", "Mobility", "English","Dynamism","Maturity","Comunication","Standing/Position"}
    id_groups = df.groupby('ID')
    counter = 0
    for id_val, group in id_groups:
        if len(group) <= 1:
            continue

        differing_cols = []
        for col in df.columns:
            if col in ignore_columns or col == 'ID':
                continue
            unique_vals = group[col].dropna().unique()
            if len(unique_vals) > 1:
                differing_cols.append((col, unique_vals))

        if differing_cols:
            if counter > 5:
                print('\n...')
                break
            else:
                counter += 1
            print(f"\nID: {id_val}")
            for col, vals in differing_cols:
                print(f"  → Column '{col}' differs: {[float(v) for v in list(vals)]}")

find_differences_by_id(df)


In [ ]:
def aggregate_group(group):
    for col in group.columns:
        if col == 'ID':
            continue  
        values = group[col].dropna().unique()
        if len(values) == 0:
            continue 
        elif len(values) == 1:
            group[col] = values[0]
        else:
            if np.issubdtype(group[col].dropna().dtype, np.number):
                avg_value = group[col].dropna().astype(float).mean()
                group[col] = avg_value
            else:
                string_values = [str(v).strip() for v in values]
                filtered_values = [v for v in string_values if v]
                combined_string = "|".join(str(v) for v in filtered_values)
                group[col] = combined_string
    return group

def aggregate_all_records(df):
    df_cleaned = df.drop(columns=['event_feedback', 'event_type__val']).drop_duplicates().reset_index(drop=True)

    grouped = df_cleaned.groupby('ID')

    groups_with_multiple = grouped.filter(lambda x: len(x) > 1)

    fixed_groups = groups_with_multiple.groupby('ID', group_keys=False).apply(aggregate_group).drop_duplicates().reset_index(drop=True)
    groups_with_single = grouped.filter(lambda x: len(x) == 1)
    final_df = pd.concat([fixed_groups, groups_with_single], ignore_index=True)

    print(f"Original number of records: {len(df['ID'])}")
    print(f"Aggregated number of records: {len(final_df['ID'])}")
    return final_df


final_df = aggregate_all_records(df)


In [ ]:
def clean_aggregated_string_column(val):
    if isinstance(val, str) and '|' in val:
        parts = val.split('|')
        filtered_parts = [p.strip() for p in parts if p.strip()]

        if not filtered_parts:
            return '' 
        elif len(filtered_parts) == 1:
            return filtered_parts[0]
        else:
            return '|'.join(filtered_parts)
    else:
        return val

def clean_aggregated_string_columns(df):
    df_cleaned = df.copy() 
    string_cols = df_cleaned.select_dtypes(include=['object', 'string']).columns

    print(f"Applying cleaning to columns: {list(string_cols)}")

    for col in string_cols:
        df_cleaned[col] = df_cleaned[col].map(clean_aggregated_string_column, na_action='ignore')

    return df_cleaned
final_df = clean_aggregated_string_columns(final_df)

### Residence

In [ ]:
with open("city_mapping.json", "r", encoding="utf-8") as f:
    city_mapping = json.load(f)
def city_transform(city):
    if city.strip().upper() in city_mapping:
        city = city_mapping[city.strip().upper()]
    else:
        city = ' '.join([c.capitalize() if c.upper() not in ['DI','IN','DEL','A'] else c.lower() for c in city.split()])
    return city

def parse_residence(residence):
    try:
        parts = residence.split('»')
    except:
        return pd.Series([None, None, None, None, False])
    city = parts[0].strip()
    if len(parts) < 2 or '~' not in parts[1]:
        italian_residence = (city.upper() == 'ITALY')  
 
        return pd.Series([city.upper(), None, None, None, italian_residence])
    subparts = parts[1].split('~')
    province = subparts[0].strip()
    region = subparts[1].strip() if len(subparts) > 1 else None
    if province == '(COUNTRY)' or province == '(STATE)':
        country = city
    else:
        country = 'ITALY'

    country = country.capitalize()
    region = region.capitalize()
    province = province.capitalize()
    city_italian_name = city_transform(city)

    if country.upper() == 'ITALY':
        italian_residence = True
    else:
        italian_residence = False
        region = None
        province = None
        city = None
    
    return pd.Series([country, region, province, city_italian_name, city, italian_residence])

final_df[['Residence Country', 'Residence Italian Region', 'Residence Italian Province', 'Residence Italian City IT', 'Residence Italian City EN', 'Italian Residence']] = final_df['Residence'].apply(parse_residence)

european_countries = {
    'ALBANIA', 'AUSTRIA', 'BELARUS', 'BELGIUM', 'BULGARIA', 'CROATIA', 'CZECH REPUBLIC',
    'FRANCE', 'GERMANY', 'GREECE', 'LITHUANIA', 'MALTA', 'MONACO', 'NETHERLANDS',
    'PORTUGAL', 'REPUBLIC OF POLAND', 'ROMANIA', 'RUSSIAN FEDERATION', 'SAN MARINO',
    'SERBIA AND MONTENEGRO', 'SLOVAKIA', 'SPAIN', 'SWEDEN', 'SWITZERLAND', 'UKRAINE',
    'GREAT BRITAIN-NORTHERN IRELAND', 'YUGOSLAVIA', 'ITALY','TÜRKIYE', 'USSR'
}
final_df['European Residence'] = final_df['Residence Country'].apply(lambda x: x.upper() in european_countries if pd.notna(x) else False)
country_mapping = {
    "GREAT BRITAIN-NORTHERN IRELAND": "UNITED KINGDOM",
    "REPUBLIC OF POLAND":           "POLAND",
    "UNITED STATES OF AMERICA" : "UNITED STATES",
    "TÜRKIYE" : "TURKEY", "SERBIA AND MONTENEGRO": "SERBIA","YUGOSLAVIA": "SERBIA", "USSR": "RUSSIA", "CHINA PEOPLE'S REPUBLIC": "CHINA",
    "SOUTH AFRICAN REPUBLIC": "SOUTH AFRICA", "RUSSIAN FEDERATION": "RUSSIA",
}


final_df['Residence Country'] = (
    final_df['Residence Country']
      .astype(str)                   
      .str.upper()                   
      .replace(country_mapping)      
)


In [ ]:
print(f'Assumption Headquarters Values: {final_df["Assumption Headquarters"].unique()}\nAkkodis Headquarters Values: {final_df["Akkodis headquarters"].unique()}')

In [ ]:
city_name_mapping = {
    "Toasts": "Brindisi",
    "The Eagle": "L'Aquila",
}
final_df['Assumption Headquarters'] = final_df['Assumption Headquarters'].replace(city_name_mapping)
final_df['Akkodis headquarters'] = final_df['Akkodis headquarters'].replace(city_name_mapping)
print(f'Assumption Headquarters Values: {final_df["Assumption Headquarters"].unique()}\nAkkodis Headquarters Values: {final_df["Akkodis headquarters"].unique()}')

In [ ]:
country_coords = {}
country_file_path = '../countries.csv'

if os.path.exists(country_file_path):
    try:
        countries_df = pd.read_csv(country_file_path)
        country_coords = {
            row['name'].upper(): {
                'latitude': row['latitude'],
                'longitude': row['longitude']
            }
            for _, row in countries_df.iterrows()
        }
        print(f"Successfully loaded country data from '{country_file_path}'.")
    except FileNotFoundError:
        print(f"Error: '{country_file_path}' not found.")
    except pd.errors.EmptyDataError:
        print(f"Error: '{country_file_path}' is empty.")
    except pd.errors.ParserError:
        print(f"Error: Could not parse '{country_file_path}'. Check CSV format.")
    except Exception as e:
        print(f"An unexpected error occurred while reading '{country_file_path}': {e}")
else:
    print(f"Warning: '{country_file_path}' not found. Country lookups will not be possible.")

cities_file = '../simplemaps_worldcities_basicv1.90/worldcities.csv'
if os.path.exists(cities_file):
    cities = pd.read_csv(cities_file)
    print(f"Loaded cities data from '{cities_file}', {len(cities)} rows.")
else:
    raise FileNotFoundError(f"Cities file not found at '{cities_file}'")


In [ ]:
def get_headquarter_coordinates(city_name):
    """
    Returns (latitude, longitude) for a headquarters city in Italy.
    Looks up in the local 'cities' DataFrame only.
    """

    if pd.isna(city_name) or not city_name:
        return None, None

    mask = (
        (cities['city_ascii'].str.upper() == str(city_name).upper()) &
        (cities['iso2'] == 'IT')
    )
    if mask.any():
        row = cities.loc[mask].iloc[0]
        return row['lat'], row['lng']
    else:
        print(f"Headquarters city '{city_name}' not found in Italy.")
        return None, None
final_df[['Assumption HQ Lat', 'Assumption HQ Lng']] = final_df['Assumption Headquarters'].apply(
    lambda x: pd.Series(get_headquarter_coordinates(x))
)

final_df[['Akkodis HQ Lat', 'Akkodis HQ Lng']] = final_df['Akkodis headquarters'].apply(
    lambda x: pd.Series(get_headquarter_coordinates(x))
)

In [ ]:
def get_city_coordinates(city_en, city_it):
    """
    Returns (latitude, longitude) for an Italian city by looking it up in the 'cities' DataFrame.
    If not found or not an Italian city, falls back to the Back4App API.
    """
    if pd.isna(city_en) or not city_en:
        return None, None

    mask = (
        (cities['city_ascii'].str.upper() == str(city_en).replace("'","").strip().upper()) &
        (cities['iso2'] == 'IT')
    )
    if mask.any():
        row = cities.loc[mask].iloc[0]
        return row['lat'], row['lng']
    mask = (
        (cities['city_ascii'].str.upper() == str(city_it).replace("'","").strip().upper()) &
        (cities['iso2'] == 'IT')
    )
    if mask.any():
        row = cities.loc[mask].iloc[0]
        return row['lat'], row['lng']
    
    city_name_str = str(city_it)
    try:
        where = urllib.parse.quote_plus(json.dumps({"name": city_name_str}))
        url = (
            'https://parseapi.back4app.com/classes/City'
            f'?limit=1&keys=name,location&where={where}'
        )
        headers = {
            'X-Parse-Application-Id': 'rPfDpoNwAXlUjYrLAYtkVa6HXYcorAOJ9pefs00V',
            'X-Parse-Master-Key': 'rpXD45YgCcmIyLf13fwUsguY9hRPaiH4xaIPsQLT'
        }
        resp = requests.get(url, headers=headers, timeout=10)
        resp.raise_for_status()
        results = resp.json().get('results', [])
        if results:
            loc = results[0].get('location', {})
            return loc.get('latitude'), loc.get('longitude')
    except requests.RequestException:
        pass
    city_name_str = str(city_en)
    try:
        where = urllib.parse.quote_plus(json.dumps({"name": city_name_str}))
        url = (
            'https://parseapi.back4app.com/classes/City'
            f'?limit=1&keys=name,location&where={where}'
        )
        headers = {
            'X-Parse-Application-Id': 'rPfDpoNwAXlUjYrLAYtkVa6HXYcorAOJ9pefs00V',
            'X-Parse-Master-Key': 'rpXD45YgCcmIyLf13fwUsguY9hRPaiH4xaIPsQLT'
        }
        resp = requests.get(url, headers=headers, timeout=10)
        resp.raise_for_status()
        results = resp.json().get('results', [])
        if results:
            loc = results[0].get('location', {})
            return loc.get('latitude'), loc.get('longitude')
    except requests.RequestException:
        pass
    print(f"City '{city_it}' not found in API response.")
    print(f"City '{city_en}' not found in API response.")
    return None, None

def get_location_coordinates(row):
    """
    For each row, attempts:
      1) Italian city lookup via `cities` DataFrame
      2) Country lookup via `countries.csv`
      3) API fallback (already inside get_city_coordinates)
    Returns a Series [latitude, longitude].
    """
    city_it = row.get('Residence Italian City IT')
    city_en = row.get('Residence Italian City EN')
    country = row.get('Residence Country')
    lat, lng = None, None
    if city_en and city_en.lower() != 'italy':
        lat, lng = get_city_coordinates(city_en, city_it)
    
    if lat is None or lng is None:
        cu = str(country).upper() if pd.notna(country) else None
        if cu and cu in country_coords:
            lat = country_coords[cu]['latitude']
            lng = country_coords[cu]['longitude']
        else:
            info = []
            if pd.notna(city_it):   info.append(f"city '{city_it}'")
            if pd.notna(country): info.append(f"country '{cu}'")
            print(f"Coordinates not found for {' and '.join(info)}.")

    return pd.Series({'Latitude': lat, 'Longitude': lng})


tqdm.pandas(desc="Geocoding rows") 

final_df[['Residence Lat', 'Residence Lon']] = final_df.progress_apply(get_location_coordinates, axis=1)



### Save Preprocessed Dataframe

In [ ]:
final_df.to_csv('preprocessed_df.csv', index=False)

## Create dataset

### Load Preprocessed Dataframe

In [ ]:
final_df = pd.read_csv('preprocessed_df.csv')

In [ ]:
final_df.columns

### Custom Similarity Features

In [ ]:
candidate_columns = [
    "Sex",
    "Age Range",
    "Protected category",
    "Italian Residence",
    "European Residence",
    "Protected category",
    "TAG",
    "Study area",
    "Study Title",
    "Years Experience",
    "Sector",
    "Last Role",
    "Current Ral",
    "Expected Ral",
    "Residence Lat",
    "Residence Lon",
    "Overall",
    "Technical Skills",
    "Standing/Position",
    "Comunication",
    "Maturity",
    "Dynamism",
    "Mobility",
    "English",
]
job_columns = [
    "Recruitment Request",
    "Job Family Hiring",
    "Job Title Hiring",
    "Job Description",
    "Candidate Profile",
    "Years Experience.1",
    "Minimum Ral",
    "Ral Maximum",
    "Study Level",
    "Study Area.1",
    "Akkodis HQ Lat",
    "Akkodis HQ Lng",
    "Assumption HQ Lat",
    "Assumption HQ Lng",
    "number_of_searches",
]

candidates = final_df[candidate_columns + ["Year of insertion", "Hired"]].copy()
jobs = final_df[job_columns + ["Year of insertion"]].copy()

valid_candidates = candidates[candidates[candidate_columns].notna().any(axis=1)].copy()
valid_jobs = jobs[jobs[job_columns].notna().any(axis=1)].drop_duplicates().copy()

final_df["candidate_text"] = final_df.apply(create_candidate_text, axis=1)
valid_jobs["job_text"] = valid_jobs.apply(create_job_text, axis=1)

model = SentenceTransformer("all-MiniLM-L6-v2")
valid_jobs = valid_jobs.reset_index()

candidate_embeddings = model.encode(
    final_df["candidate_text"].fillna("").tolist(), show_progress_bar=True
)
job_embeddings = model.encode(
    valid_jobs["job_text"].fillna("").tolist(), show_progress_bar=True
)

cos_sim_matrix = cosine_similarity(candidate_embeddings, job_embeddings)

In [ ]:
new_dataset = []

for idx, row in tqdm(final_df.iterrows(), total=len(final_df)):
    year = row['Year of insertion']
    hired = row['Hired']
    cand_text = row['candidate_text']
    full_candidate_data = row[candidate_columns].to_dict()

    same_year_mask = valid_jobs['Year of insertion'] == year
    year_job_indices = valid_jobs[same_year_mask].index.tolist()
   
    if not year_job_indices:
        print('No year')
        continue

    similarities = [cos_sim_matrix[idx][j] for j in year_job_indices]


    if hired == 1:
        job_data = row[job_columns].to_dict()
        new_dataset.append({**full_candidate_data, **job_data, "Hired": 1})
        low_sim_indices = sorted(zip(year_job_indices, similarities), key=lambda x: x[1])[:3]
        low_sample_idx = random.choice(low_sim_indices)[0]
        job_data_neg = valid_jobs.loc[low_sample_idx][job_columns].to_dict()
        new_dataset.append({**full_candidate_data, **job_data_neg, "Hired": 0})

    elif hired == 0:
        low_sim_indices = sorted(zip(year_job_indices, similarities), key=lambda x: x[1])[:3]
        high_sim_indices = sorted(zip(year_job_indices, similarities), key=lambda x: x[1], reverse=True)[:8]
        low_sample_idx = random.choice(low_sim_indices)[0]
        high_sample_idx = random.choice(high_sim_indices)[0]
        
        job_data_low = valid_jobs.loc[low_sample_idx][job_columns].to_dict()
        job_data_high = valid_jobs.loc[high_sample_idx][job_columns].to_dict()
        new_dataset.append({**full_candidate_data, **job_data_low, "Hired": 0})
        new_dataset.append({**full_candidate_data, **job_data_high, "Hired": 0})


dataset = pd.DataFrame(new_dataset)
dataset.head()

In [ ]:
cat_order = {
    'Age Range': ['< 20 years', '20 - 25 years', '26 - 30 years', '31 - 35 years', '36 - 40 years', '40 - 45 years', '> 45 years'],
    'Years Experience': ['[0]', '[0-1]', '[1-3]', '[3-5]', '[5-7]', '[7-10]', '[+10]'],
    'Years Experience.1': ['[0]', '[0-1]',  '[1-3]', '[3-5]', '[5-7]', '[7-10]','[+10]'],
    'Sex': ['Female','Male'],
    'Study Level':[
        "Middle school diploma",
        "High school graduation",
        "Professional qualification",
        "Three-year degree",
        "Five-year degree",
        "master's degree",
        "Doctorate"
    ], 
    'Study Title':[
        "Middle school diploma",
        "High school graduation",
        "Professional qualification",
        "Three-year degree",
        "Five-year degree",
        "master's degree",
        "Doctorate"
    ]
}

for col, order in cat_order.items():
    if col in dataset.columns:
        dataset[col+'_int'] = pd.Categorical(dataset[col], categories=order, ordered=True)
        dataset[col+'_int'] = dataset[col+'_int'].cat.codes.replace(-1, pd.NA)
dataset['Study Level_int'] = (
dataset['Study Level_int']
.astype('Int64') 
)
dataset['Years Experience.1_int'] = dataset['Years Experience.1_int'].fillna(4)
dataset['experience_match_score'] = calculate_experience_match_score(dataset)

dataset['current_salary_fit_score'] = calculate_salary_fit_score(dataset, is_expected=False)
dataset['expected_salary_fit_score'] = calculate_salary_fit_score(dataset, is_expected=True)
dataset['study_title_score'] = calculate_study_title_score(dataset)
dataset['professional_similarity_score'] = calculate_professional_similarity_score(dataset)
dataset['study_area_score'] = calculate_study_area_score(dataset)

dataset['Distance Residence - Akkodis HQ'] = dataset.apply(
    lambda row: calculate_distance(
        (row['Residence Lat'], row['Residence Lon']),
        (row['Akkodis HQ Lat'], row['Akkodis HQ Lng'])
    ),
    axis=1
)

dataset['Distance Residence - Assumption HQ'] = dataset.apply(
    lambda row: calculate_distance(
        (row['Residence Lat'], row['Residence Lon']),
        (row['Assumption HQ Lat'], row['Assumption HQ Lng'])
    ),
    axis=1
)

In [ ]:
dataset = prepare_nlp_text_columns(dataset)

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

def compute_general_similarity_score(df: pd.DataFrame) -> pd.Series:
    embedding_cache = {}

    def get_embedding(text):
        if text in embedding_cache:
            return embedding_cache[text]
        embedding = model.encode(text, convert_to_tensor=True)
        embedding_cache[text] = embedding
        return embedding

    def similarity(row):
        candidate_text = row.get('candidate_text')
        job_text = row.get('job_text')

        if not candidate_text or not job_text:
            return np.nan

        emb_a = get_embedding(candidate_text)
        emb_b = get_embedding(job_text)
        return float(util.cos_sim(emb_a, emb_b))

    return df.apply(similarity, axis=1)
dataset['general_similarity_score'] = compute_general_similarity_score(dataset)


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

cross_model = CrossEncoder("cross-encoder/ms-marco-TinyBERT-L2-v2", device=device)
def compute_similarity_with_prompt(df: pd.DataFrame, batch_size: int = 256) -> pd.Series:
    valid_df = df[['candidate_text', 'job_text']].dropna()
    pairs = valid_df.values.tolist()
    
    tokenizer = cross_model.tokenizer
    prompted_pairs = []

    for cand, job in pairs:
        prompted_pairs.append((
            job,
            cand
        ))

 
    scores = cross_model.predict(prompted_pairs, batch_size=batch_size, show_progress_bar=True)
    
    result = pd.Series(np.nan, index=df.index, dtype=np.float32)
    result.loc[valid_df.index] = scores
    return result

dataset['general_similarity_score_cross'] = compute_similarity_with_prompt(dataset)

In [ ]:
tfidf = TfidfVectorizer(
    max_features=5000,   
    stop_words='english'
)

combined_text = pd.concat([dataset['candidate_text'], dataset['job_text']])
tfidf.fit(combined_text.fillna(""))

candidate_tfidf_dense = tfidf.transform(dataset['candidate_text'].fillna("")).toarray()
job_tfidf_dense = tfidf.transform(dataset['job_text'].fillna("")).toarray()

tfidf_sim_matrix = cosine_similarity(candidate_tfidf_dense, job_tfidf_dense)

dataset['general_similarity_score_tfidf'] = tfidf_sim_matrix.max(axis=1)


In [ ]:
columns_to_keep = [
    "Sex_int",
    "Protected category",
    "Overall",
    "Technical Skills",
    "Standing/Position",
    "Comunication",
    "Maturity",
    "Dynamism",
    "Mobility",
    "English",
    "Hired",
    "Italian Residence",
    "European Residence",
    "Age Range_int",
    "experience_match_score",
    "Years Experience_int",
    "Years Experience.1_int",
    "current_salary_fit_score",
    "Current Ral",
    "Expected Ral",
    "Minimum Ral",
    "Ral Maximum",
    "expected_salary_fit_score",
    "study_title_score",
    "Study Level_int",
    "Study Title_int",
    "professional_similarity_score",
    "study_area_score",
    "general_similarity_score",
    "general_similarity_score_tfidf",
    "general_similarity_score_cross",
    "number_of_searches",
    "Distance Residence - Akkodis HQ",
    "Distance Residence - Assumption HQ",
]

### Dataset Analysis

In [ ]:
df_corr = dataset[columns_to_keep].copy()

bool_cols = df_corr.select_dtypes(include='bool').columns
df_corr[bool_cols] = df_corr[bool_cols].astype(int)

df_corr_numeric = df_corr.select_dtypes(include=[np.number])

plt.figure(figsize=(20, 10))
sns.heatmap(df_corr_numeric.corr(), annot=False, 
              cmap="coolwarm", center=0, square=True)
plt.title("Full Correlation Matrix")
plt.tight_layout()
plt.show()

correlations = df_corr_numeric.corr()['Hired'].drop('Hired').sort_values()

plt.figure(figsize=(10, 6))
sns.stripplot(y=correlations.values, x=correlations.index, color='darkgreen', size=10)
plt.axhline(0, color='gray', linestyle='--')
plt.title("Feature Correlations with 'Hired'")
plt.ylabel("Correlation Coefficient")
plt.xlabel("Features")
plt.xticks(rotation=45, ha='right')
plt.grid(True, axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


In [ ]:
print("Hiring Rate by Sex:")
distribution_by_sex = dataset.groupby('Sex')['Hired'].mean()
print(distribution_by_sex)

print("Hiring Rate by European Residence:")
distribution_by_eu_residence = dataset.groupby('European Residence')['Hired'].mean()
print(distribution_by_eu_residence)

print("Hiring Rate by Italian Residence:")
distribution_by_it_residence = dataset.groupby('Italian Residence')['Hired'].mean()
print(distribution_by_it_residence)

print("\nHiring Rate by Age Range:")
distribution_by_age = dataset.groupby('Age Range')['Hired'].mean()
print(distribution_by_age)

print("\nHiring Rate by Protected Category:")
distribution_by_category = dataset.groupby('Protected category')['Hired'].mean()
print(distribution_by_category)

sns.barplot(x='Sex', y='Hired', data=dataset, estimator=np.mean, palette='Set2')
plt.title("Hiring Rate by Sex")
plt.ylim(0, 1.2 * max(distribution_by_sex))
plt.ylabel("Proportion Hired")
plt.show()

sns.barplot(x='European Residence', y='Hired', data=dataset, estimator=np.mean, palette='Set2')
plt.title("Hiring Rate by European Residence")
plt.ylim(0, 1.2 * max(distribution_by_eu_residence))
plt.ylabel("Proportion Hired")
plt.show()

sns.barplot(x='Italian Residence', y='Hired', data=dataset, estimator=np.mean, palette='Set2')
plt.title("Hiring Rate by Italian Residence")
plt.ylim(0, 1.2 * max(distribution_by_it_residence))
plt.ylabel("Proportion Hired")
plt.show()

sns.barplot(x='Age Range', y='Hired', data=dataset, estimator=np.mean, palette='Set3')
plt.title("Hiring Rate by Age Range")
plt.ylim(0, 1.3 * max(distribution_by_age))
plt.xticks(rotation=45)
plt.ylabel("Proportion Hired")
plt.show()

sns.barplot(x='Protected category', y='Hired', data=dataset, estimator=np.mean, palette='Set1')
plt.title("Hiring Rate by Protected Category")
plt.ylim(0, 1.2 * max(distribution_by_category))
plt.ylabel("Proportion Hired")
plt.show()

In [ ]:
summary_sex = dataset.groupby('Sex')['Hired'].agg(['mean', 'count']).rename(columns={'mean': 'Hiring Rate', 'count': 'Number of Candidates'})
print("\nHiring Rate and Count by Sex:\n", summary_sex)

summary_eu_residence = dataset.groupby('European Residence')['Hired'].agg(['mean', 'count']).rename(columns={'mean': 'Hiring Rate', 'count': 'Number of Candidates'})
print("\nHiring Rate and Count by European Residence:\n", summary_eu_residence)

summary_it_residence = dataset.groupby('Italian Residence')['Hired'].agg(['mean', 'count']).rename(columns={'mean': 'Hiring Rate', 'count': 'Number of Candidates'})
print("\nHiring Rate and Count by Italian Residence:\n", summary_it_residence)

summary_age = dataset.groupby('Age Range')['Hired'].agg(['mean', 'count']).rename(columns={'mean': 'Hiring Rate', 'count': 'Number of Candidates'}).sort_index()
print("\nHiring Rate and Count by Age Range:\n", summary_age)

summary_protected = dataset.groupby('Protected category')['Hired'].agg(['mean', 'count']).rename(columns={'mean': 'Hiring Rate', 'count': 'Number of Candidates'})
print("\nHiring Rate and Count by Protected Category:\n", summary_protected)


In [ ]:
intersection = dataset.groupby(['Sex', 'Age Range'])['Hired'].mean().unstack()
print("\nHiring Rate by Sex and Age Range:")
print(intersection)

sns.heatmap(intersection, annot=True, cmap='Blues', fmt=".2f")
plt.title("Hiring Rates by Sex and Age Range")
plt.ylabel("Sex")
plt.xlabel("Age Range")
plt.show()

In [ ]:
p_selected_female = dataset[dataset['Sex'] == 'Female']['Hired'].mean()
p_selected_male = dataset[dataset['Sex'] == 'Male']['Hired'].mean()

disparate_impact = p_selected_female / p_selected_male
print("\nDisparate Impact Ratio (Female vs Male):", round(disparate_impact, 3))

**Analysis of Hiring Rates**

**1. Gender (Sex)**
Females are hired at a significantly higher rate than males, suggesting a possible organizational emphasis on gender diversity or a potential bias favoring female candidates.

**2. Age Range**
Hiring rates increase with age, peaking between 31–45 years, indicating a clear preference for mid-career professionals with more experience. Younger candidates, especially under 26, face notably lower hiring chances.

**3. European Residence**
Candidates residing in Europe are far more likely to be hired, which may reflect logistical preferences, legal work eligibility, or alignment with company locations and operations.

**4. Italian Residence**
There is a strong hiring bias toward candidates living in Italy. This suggests the organization prefers local hires, potentially to reduce relocation costs or due to legal/employment constraints.

**5. Protected Category**
No meaningful difference in hiring rates between protected and non-protected groups was found. However, due to the very small sample of protected category candidates, no reliable conclusion can be drawn.

### Drop Nan Values

In [ ]:
df = dataset.copy()

print(f"{'Column':30} | {'Rows Before':10} | {'Rows After':10} | {'Hired Before':12} | {'Hired After':11} | {'% Hired Before':14} | {'% Hired After':13}")
print("-" * 105)

rows_before = len(df)
hired_before = df['Hired'].sum()
perc_hired_before = hired_before / rows_before * 100

for col in columns_to_keep:
    df_dropped = df.dropna(subset=[col])
    
    rows_after = len(df_dropped)
    hired_after = df_dropped['Hired'].sum()
    
    perc_hired_after = hired_after / rows_after * 100 if rows_after > 0 else 0
    
    print(f"{col:30} | {rows_before:<10} | {rows_after:<10} | {hired_before:<12} | {hired_after:<11} | {perc_hired_before:<14.2f} | {perc_hired_after:<13.2f}")


In [ ]:
df_cleaned = dataset.dropna(subset=[ 'professional_similarity_score', ])

print(f"Original data shape: {dataset.shape,(dataset['Hired']==True).sum()}")
print(f"Cleaned data shape: {df_cleaned.shape,(df_cleaned['Hired']==True).sum()}")


#### Save Cleaned Dataset

In [ ]:
df_cleaned.to_csv('cleaned_dataset.csv', index=False)

In [ ]:
dataset.to_csv('full_dataset.csv', index=False)